In [2]:
import math
import numpy as np
import random

In [260]:
class Value():
    def __init__(self, data, _children = (), _op = "", _label = ""):
        self.data = data
        self._prev = _children
        self._op = _op
        self._label = _label
        self.grad = 0.0
        self._backward = lambda:None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value((self.data + other.data), (self, other), "+")
        def _backwards():
            self.grad += out.grad
            other.grad += out.grad
        
        out._backward = _backwards
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value((self.data * other.data), (self, other), "*")
        def _backwards():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backwards
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value((self.data**other), (self, ), "pow")
        def _backwards():
            self.grad += (other * self.data ** (other-1)) * out.grad
        
        out._backward = _backwards
        return out

    def __exp__(self):
        out = Value((math.exp(self.data)), (self, ), "exp")
        def _backwards():
            self.grad += out.data * out.grad
        out._backward = _backwards
        return out

    def tanh(self):
        a = math.exp(2*self.data)
        b = (a-1)/(a+1)
        out = Value(b, (self, ), "tanh")
        def _backwards():
            self.grad += (1 - b**2) * out.grad
        out._backward = _backwards
        return out

    def backward(self):
        topological_sorted = []
        visited_set = set()
        def topo_sort(self):
            if self not in visited_set:
                visited_set.add(self)
                for child in self._prev:
                    topo_sort(child)
                topological_sorted.append(self)
        topo_sort(self)
        self.grad = 1.0
        for node in reversed(topological_sorted):
            node._backward()

    def __neg__(self): # negative
        return self * -1

    def __radd__(self, other): # add
        return self + other

    def __sub__(self, other): # substract
        return self + (-other)

    def __rmul__(self, other): # *
        return self * other

    def __truediv__(self, other): # divide
        return self * other**(-1)

    def __repr__(self): # fancy wrapper
        return f"Value: {self.data}"


In [261]:
class Neuron():
    def __init__(self, number_inputs):
        self.w = [Value((random.uniform(-1,1))) for _ in range(number_inputs)]
        self.b = Value((random.uniform(-1,1)))

    def __call__(self, x):
        # x*w + b
        zipped = (zip(self.w, x))
        return sum([x1*w1 for w1, x1 in zipped],self.b)

    def parameters(self):
        return self.w + [self.b]

class Layer():
    def __init__(self, number_inputs, number_neurons):
        self.neurons = [Neuron(number_inputs) for _ in range(number_neurons)]

    def __call__(self, x):
        lst = []
        for neuron in self.neurons:
            lst.append(neuron(x))
        return lst[0] if len(lst)==1 else lst

    def parameters(self):
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())
        return params

class MLP():
    def __init__(self, number_inputs, number_neurons):
        self.size = [number_inputs] + number_neurons
        self.layers = [Layer(self.size[i], self.size[i+1]) for i in range(len(number_neurons))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params

In [262]:
# forward pass
def forward_pass(model, x, y):
    y_pred = [model(xi) for xi in x]
    return sum(((y_pred_i-yi)**2) for y_pred_i, yi in zip(y_pred, y))

# gradient set to 0
def zero_grads(model):
    for param in model.parameters():
        param.grad = 0.0

# backward pass
def backward_pass(loss):
    return loss.backward()

def update_data(model, learning_rate):
    for param in model.parameters():
        param.data += -learning_rate*param.grad

def train(model, x, y, learning_rate, batch):
    for i in range(batch):
        loss = forward_pass(model, x, y)
        zero_grads(model)
        backward_pass(loss)
        update_data(model, learning_rate)
        print(f"Step: {i}. Loss: {loss.data}")


In [263]:
### DATA

# given 4 inputs:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]
# desired outputs:
ys = [1.0, -1.0, -1.0, 1.0]

### MODEL

number_inputs = 3
number_neurons = [3, 4, 1]
model = MLP(number_inputs, number_neurons)

### PARAMS

lr = 0.01
batch_size = 20

In [264]:
train(model, xs, ys, lr, batch_size)

Step: 0. Loss: 12.982863165653018
Step: 1. Loss: 2.8517312956534875
Step: 2. Loss: 2.003028157449867
Step: 3. Loss: 1.6314536761094414
Step: 4. Loss: 1.387868907153219
Step: 5. Loss: 1.185052281157617
Step: 6. Loss: 1.007244298094992
Step: 7. Loss: 0.8508498791040373
Step: 8. Loss: 0.7140199009256708
Step: 9. Loss: 0.5952494928523353
Step: 10. Loss: 0.49311577300992837
Step: 11. Loss: 0.4061885011879709
Step: 12. Loss: 0.3329995805939212
Step: 13. Loss: 0.27204770048431626
Step: 14. Loss: 0.22182505532807972
Step: 15. Loss: 0.180855220130695
Step: 16. Loss: 0.14773294855401126
Step: 17. Loss: 0.1211589167559466
Step: 18. Loss: 0.09996509514954197
Step: 19. Loss: 0.0831290359789762
